In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd
import gc
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback)

In [ ]:
df = pd.read_csv("balanced_reddit_posts_dataset2.csv")
df

,clean_comment,category,clean_text,tokens,final_text
0,mandybao 152 看见了一个熟悉的论调，西部人民还在吃草\n,0,mandybao,['mandybao'],mandybao
1,its funny that congressi now introduces himsel...,-1,its funny that congressi now introduces himsel...,"['funny', 'congressi', 'introduces', 'liberal'...",funny congressi introduces liberal exactly lib...
2,are you insanely unhappy life india has surviv...,-1,are you insanely unhappy life india has surviv...,"['insanely', 'unhappy', 'life', 'india', 'surv...",insanely unhappy life india survived much wors...
3,downvote karna tha par upvote hogaya,0,downvote karna tha par upvote hogaya,"['downvote', 'karna', 'tha', 'par', 'upvote', ...",downvote karna tha par upvote hogaya
4,part 20161114,0,part,['part'],part
...,...,...,...,...,...
24745,the long press the recents button for multiwi...,1,the long press the recents button for multiwin...,"['long', 'press', 'recents', 'button', 'multiw...",long press recents button multiwindow gone acc...
24746,better have dumb who has the sense listen the...,-1,better have dumb who has the sense listen the ...,"['better', 'dumb', 'sense', 'listen', 'experts...",better dumb sense listen experts field rather ...
24747,the wife the other hand,-1,the wife the other hand,"['wife', 'hand']",wife hand
24748,hello and thank you banerjee does gbs has spec...,0,hello and thank you banerjee does gbs has spec...,"['hello', 'thank', 'banerjee', 'gbs', 'special...",hello thank banerjee gbs specialized traveling...


In [ ]:
texts = df["clean_text"].astype(str)
labels = df["category"]

label_mapping = {-1: 0, 0: 1, 1: 2}
y = df["category"].map(label_mapping).astype(int).values

num_labels = 3

In [ ]:
train_texts_full, test_texts, train_labels_full, test_labels = train_test_split(
    texts, y, test_size=0.2, random_state=42, stratify=y
)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts_full, train_labels_full, test_size=0.2,
    random_state=42, stratify=train_labels_full
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=512
    )

train_encodings = tokenize(train_texts)
val_encodings   = tokenize(val_texts)
test_encodings  = tokenize(test_texts)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_dataset = SentimentDataset(train_encodings, train_labels)
val_dataset   = SentimentDataset(val_encodings, val_labels)
test_dataset  = SentimentDataset(test_encodings, test_labels)

In [ ]:
def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

In [ ]:
def run_experiment(batch_size, lr, epochs=4):
    print(f"\nRunning mBERT | batch={batch_size} | lr={lr}")
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-multilingual-cased",
        num_labels=num_labels)

In [ ]:
def run_experiment(batch_size, lr, epochs=4):

    print(f"\nRunning mBERT | batch={batch_size} | lr={lr}")

    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-multilingual-cased",
        num_labels=num_labels
    )

    training_args = TrainingArguments(
        output_dir=f"./mbert_bs{batch_size}_lr{lr}",

        eval_strategy="epoch",
        save_strategy="epoch",

        num_train_epochs=epochs,

        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,

        learning_rate=lr,
        weight_decay=0.1,

        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        max_grad_norm=1.0,

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        logging_steps=50,
        fp16=torch.cuda.is_available(),
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    metrics = trainer.evaluate()

    f1 = metrics["eval_f1"]
    loss = metrics["eval_loss"]

    print(f"Validation F1 = {f1:.4f}")

    del trainer
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return f1, loss

In [ ]:
batch_sizes = [16, 32]
learning_rates = [2e-5, 3e-5, 5e-5]

best_f1 = 0
best_params = None

for bs in batch_sizes:
    for lr in learning_rates:
        f1, loss = run_experiment(bs, lr, epochs=4)

        if f1 > best_f1:
            best_f1 = f1
            best_params = (bs, lr)

print("\nBEST CONFIG:", best_params, "F1:", round(best_f1, 4))


Running mBERT | batch=16 | lr=2e-05


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.460127,0.394748,0.859596,0.859534
2,0.275957,0.321961,0.904040,0.904405
3,0.215335,0.283324,0.924747,0.924904
4,0.117294,0.326236,0.928030,0.928260


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Validation F1 = 0.9169

BEST CONFIG: (16, 2e-05) F1: 0.9249


In [ ]:
best_bs, best_lr = best_params

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir="./best_mbert_model",

    eval_strategy="epoch",
    save_strategy="epoch",

    num_train_epochs=4,

    per_device_train_batch_size=best_bs,
    per_device_eval_batch_size=best_bs,

    learning_rate=best_lr,
    weight_decay=0.1,

    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.828276,0.406849,0.860101,0.859679
2,0.314463,0.280407,0.909596,0.909722
3,0.202925,0.313657,0.913889,0.914178
4,0.127385,0.352646,0.918182,0.918494


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3960, training_loss=0.3199414397730972, metrics={'train_runtime': 2161.3485, 'train_samples_per_second': 29.315, 'train_steps_per_second': 1.832, 'total_flos': 1.667086614724608e+16, 'train_loss': 0.3199414397730972, 'epoch': 4.0})

In [ ]:
pred_test = trainer.predict(test_dataset)
preds_test = np.argmax(pred_test.predictions, axis=1)

class_names = ["Negative", "Neutral", "Positive"]

print(classification_report(test_labels, preds_test, target_names=class_names))

              precision    recall  f1-score   support

    Negative       0.87      0.89      0.88      1650
     Neutral       0.95      0.93      0.94      1650
    Positive       0.89      0.89      0.89      1650

    accuracy                           0.90      4950
   macro avg       0.91      0.90      0.90      4950
weighted avg       0.91      0.90      0.90      4950



In [ ]:
cm = confusion_matrix(test_labels, preds_test)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Test Set")
plt.show()

In [ ]:
y_test_bin = label_binarize(test_labels, classes=range(num_labels))
y_score = pred_test.predictions

plt.figure(figsize=(8,6))
for i in range(num_labels):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{le.classes_[i]} (AUC = {roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC Curve - Test Set")
plt.legend(loc="lower right")
plt.show()